**DETECCIÓN Y LECTURA DE MATRÍCULAS EN VÍDEO CON RT-DETRv2 Y PADDLEOCR-VL, UN VLM**

En este cuaderno se detectan personas y vehículos con RT-DETRv2, se les asigna un identificador con ByteTrack, se localiza la matrícula de cada vehículo con el detector de matrículas (RT-DETRv2 tamaño `s`) y se lee su texto con el modelo de visión-lenguaje `PaddlePaddle/PaddleOCR-VL`.

El flujo de trabajo se estructura en:
1. **Configuración inicial**: Importaciones, clonado de RT-DETR y selección del dispositivo (GPU si está disponible).
2. **Modelo de matrículas**: Carga del detector de matrículas.
3. **Funciones de OCR con VLM**: Carga de PaddleOCR-VL y lectura de matrículas con su confianza.
4. **Procesamiento de vídeo**: Detección, seguimiento, localización de matrículas y OCR sobre un vídeo, guardando el vídeo anotado y un CSV.

**CONFIGURACIÓN INICIAL: IMPORTS Y DISPOSITIVO**

Se clona el repositorio de RT-DETR si no existe, se añade su ruta a `sys.path`, se define el identificador del VLM y se selecciona la GPU si PyTorch detecta una con CUDA.

Para que el VLM cargue hace falta tener instalada la librería `sentencepiece`. Si falta, aparece `Cannot instantiate this tokenizer from a slow version`, `get_ocr()` devuelve `None` y el vídeo se procesa sin OCR. Se soluciona una vez con `pip install sentencepiece protobuf` y reiniciando el kernel.

In [1]:
import os, sys, subprocess
import torch

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

#Cargamos el modelo y ponemos la ruta en memoria
if not os.path.exists("RT-DETR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/lyuwenyu/RT-DETR.git"], check=True)


RTDETR_PATH = os.path.abspath("RT-DETR/rtdetrv2_pytorch")
if RTDETR_PATH not in sys.path:
    sys.path.insert(0, RTDETR_PATH)

from load_rtdetr import load_rtdetr

VLM_ID = "PaddlePaddle/PaddleOCR-VL"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**MODELO DE MATRÍCULAS**

Se carga el detector de matrículas (RT-DETRv2 tamaño `s`), con una única clase (`plate`), que se aplica sobre el recorte de cada vehículo.

In [2]:
from pathlib import Path

repo = Path("RT-DETR/rtdetrv2_pytorch")

print("Configuraciones RT-DETRv2:")
for f in (repo / "configs/rtdetrv2").glob("*.yml"):
    print(f.name)

print("\nConfiguraciones de matrículas en todo el repositorio:")
for f in repo.rglob("*plate*.yml"):
    print(f)

Configuraciones RT-DETRv2:
rtdetrv2_hgnetv2_h_6x_coco.yml
rtdetrv2_hgnetv2_l_6x_coco.yml
rtdetrv2_hgnetv2_x_6x_coco.yml
rtdetrv2_r101vd_6x_coco.yml
rtdetrv2_r18vd_120e_coco.yml
rtdetrv2_r18vd_120e_plates.yml
rtdetrv2_r18vd_120e_voc.yml
rtdetrv2_r18vd_dsp_3x_coco.yml
rtdetrv2_r18vd_sp1_120e_coco.yml
rtdetrv2_r18vd_sp2_120e_coco.yml
rtdetrv2_r18vd_sp3_120e_coco.yml
rtdetrv2_r34vd_120e_coco.yml
rtdetrv2_r34vd_dsp_1x_coco.yml
rtdetrv2_r50vd_6x_coco.yml
rtdetrv2_r50vd_dsp_1x_coco.yml
rtdetrv2_r50vd_m_7x_coco.yml
rtdetrv2_r50vd_m_dsp_3x_coco.yml

Configuraciones de matrículas en todo el repositorio:
RT-DETR\rtdetrv2_pytorch\configs\dataset\plate_detection.yml
RT-DETR\rtdetrv2_pytorch\configs\rtdetrv2\rtdetrv2_r18vd_120e_plates.yml


In [3]:
PLATE_NAME = {0: "plate"}
PLATE_CLASS = PLATE_NAME.keys()
PLATE_COLOR = {0: (128, 255, 128)}

model_plate, postprocessor_plate = load_rtdetr(family="v2", size="s", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 's' en cuda


**FUNCIONES DE OCR: PADDLEOCR-VL**

Se carga el VLM una sola vez (`get_ocr`) y se definen las funciones de lectura:
- **`norm`**: Normaliza el texto a mayúsculas alfanuméricas.
- **`recortar`**: Recorta una región con margen porcentual o en píxeles, ajustada a los límites de la imagen.
- **`leer_placa`**: Reescala el recorte, lo pasa al VLM con el prompt `OCR:` y devuelve el texto y la confianza media de los tokens generados.

Se corrigió la línea `torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32`: comparaba un `torch.device` con un texto y daba siempre `False`, de modo que el modelo se cargaba siempre en `float32`. Ahora usa `DEVICE.type == "cuda"`. Si `bfloat16` diera un error de tipos en tu GPU, vuelve a `torch.float32` en esa línea.

In [4]:
from transformers import AutoModelForCausalLM, AutoProcessor
import re
import cv2
import numpy as np
from PIL import Image

# =====================================================
# GESTIÓN Y FUNCIONES OCR (PADDLEOCR-VL)
# =====================================================

ocr_global = None
_ocr_disponible = True

def get_ocr():
    """Devuelve (modelo, processor) del VLM global, inicializandolo solo la primera vez."""
    global ocr_global, _ocr_disponible

    if not _ocr_disponible:
        return None

    if ocr_global is None:
        try:
            processor = AutoProcessor.from_pretrained(
                VLM_ID,
                trust_remote_code=True
            )

            modelo = AutoModelForCausalLM.from_pretrained(
                VLM_ID,
                trust_remote_code=True,
                torch_dtype=torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
            ).to(DEVICE).eval()

            ocr_global = (modelo, processor)

        except Exception as e:
            print(f"ERROR cargando PaddleOCR-VL: {e}")
            _ocr_disponible = False
            return None

    return ocr_global


# Inicializamos el VLM globalmente
ocr_global = get_ocr()


# Normaliza texto a mayúsculas alfanuméricas
def norm(s: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", s.upper())

# Recorta una región con margen porcentual o en píxeles ajustada a los límites de la imagen
def recortar(img, box, pad=0.05, margen_px=None):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    if margen_px is not None:
        return img[max(0, int(y1 - margen_px)):min(h, int(y2 + margen_px)),
                   max(0, int(x1 - margen_px)):min(w, int(x2 + margen_px))]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    y2 = min(h, int(y2 + pad * bh))
    return img[y1:y2, x1:x2]

# Prompt de reconocimiento de texto
PROMPT_PLACA = "OCR:"

def leer_placa(crop, ocr_inst=None):
    """Devuelve (texto normalizado, confianza media) usando PaddleOCR-VL."""

    if crop is None or crop.size == 0:
        return "", 0.0

    reader = ocr_inst if ocr_inst is not None else get_ocr()

    if reader is None:
        return "", 0.0

    modelo, processor = reader

    # Escala minima garantizada tanto en alto como en ancho
    h_c, w_c = crop.shape[:2]
    esc = max(max(1.0, 64.0 / h_c), max(1.0, 200.0 / w_c))

    g = cv2.resize(
        crop, None, fx=esc, fy=esc,
        interpolation=cv2.INTER_CUBIC
    )

    g = cv2.GaussianBlur(g, (3, 3), 0)
    g = cv2.copyMakeBorder(
        g, 10, 10, 10, 10, cv2.BORDER_REPLICATE
    )

    # Convertimos el recorte a PIL
    pil = Image.fromarray(cv2.cvtColor(g, cv2.COLOR_BGR2RGB))

    # Preparacion de la entrada para PaddleOCR-VL
    mensajes = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil},
            {"type": "text", "text": PROMPT_PLACA}
        ]
    }]

    inputs = processor.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(DEVICE)

    # Inferencia
    with torch.inference_mode():
        salida = modelo.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Extraemos solamente los tokens generados
    gen = salida.sequences[:, inputs["input_ids"].shape[1]:]

    respuesta = processor.batch_decode(
        gen,
        skip_special_tokens=True
    )[0].strip()

    texto = norm(respuesta)

    if not texto or texto == "NONE":
        return "", 0.0

    # Confianza media de los tokens generados
    try:
        logp = modelo.compute_transition_scores(
            salida.sequences,
            salida.scores,
            normalize_logits=True
        )[0]

        especiales = set(processor.tokenizer.all_special_ids)

        probs = [
            float(torch.exp(lp))
            for lp, t in zip(logp, gen[0].tolist())
            if t not in especiales
        ]

        confianza = float(np.mean(probs)) if probs else 0.0

    except Exception:
        confianza = 1.0

    return texto, confianza

c:\Users\enriq\anaconda3\envs\matriculas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


**PARÁMETROS DEL VÍDEO Y MODELO DE VEHÍCULOS**

Parámetros de detección, seguimiento y OCR sobre vídeo:
- **Clases y colores**: Personas, coches, motos, autobuses y camiones (clases COCO).
- **Recortes de vehículo**: Las matrículas se buscan dentro del recorte de cada vehículo y heredan su `tracker_id`.
- **Lectura OCR**: Frecuencia, número máximo de intentos y confianza mínima para dar una lectura por buena.
- **Patrones de matrícula**: Expresiones regulares de formatos europeos para validar el texto leído.

Aquí se carga también el detector de vehículos y personas (RT-DETRv2 tamaño `l`).

In [5]:
CLASS_NAMES = {0: 'person', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}  # 0..79
BASE_CLASS = CLASS_NAMES.keys()
BASE_COLORS = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255), 5: (0, 255, 255), 7: (128, 128, 255)}

# Las matriculas se buscan dentro del recorte de cada vehiculo y heredan su tracker_id
MAX_CROPS = 8          # maximo de vehiculos por frame donde se buscan matriculas
MIN_CROP_SIDE = 48     # px: vehiculos mas pequenos que esto se ignoran
CROP_MARGIN = 0.05     # margen extra alrededor del vehiculo
PLATE_CONF = 0.3      # confianza minima del detector de matriculas
HOLD_FRAMES = 5        # frames que se mantiene la caja de matricula si no se detecta

# Parámetros visuales y de OCR de matrículas
COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35

# Parámetros del OCR sobre vídeo
OCR_CADA = 5
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 6

# Patrones sintácticos de matrículas europeas
CONS = "BCDFGHJKLMNPRSTVWXYZ"
PATRONES_MATRICULA = [re.compile(p) for p in (
    rf"\d{{4}}[{CONS}]{{3}}",              # España actual
    r"[A-Z]{1,2}\d{4}[A-Z]{1,2}",          # España antigua
    r"[A-Z]{2}\d{3}[A-Z]{2}",              # Francia / Italia
    r"[A-Z]{2}\d{2}[A-Z]{3}",              # Reino Unido
    r"[A-Z]{1,3}[A-Z]{1,2}\d{1,4}[EH]?",   # Alemania
    r"\d{2}[A-Z]{2}\d{2}",                 # Portugal
    r"[A-Z]{2}\d{2}[A-Z]{2}",              # Portugal
    r"\d{2}\d{2}[A-Z]{2}",                 # Portugal
    r"[1-9][A-Z]{3}\d{3}",                 # Bélgica
    r"\d{2}[A-Z]{3}\d",                    # Países Bajos
    r"\d[A-Z]{3}\d{2}",
    r"[A-Z]{2}\d{3}[A-Z]",
    r"\d{3}[A-Z]{2}\d",
    r"[A-Z]\d{3}[A-Z]{2}",
    r"[A-Z]{3}\d{2}[A-Z]",
    r"[A-Z]{3}\d{3}",                      # Polonia / otros
    r"[A-Z]{2}\d{5}",
)]

# Valida si el texto coincide con alguna sintaxis europea conocida
def matricula_valida(texto):
    return any(p.fullmatch(texto) for p in PATRONES_MATRICULA)

model, postprocessor = load_rtdetr(family="v2", size="l", device=DEVICE)

Cargando RT-DETR familia 'v2', tamaño 'l' en cuda
Downloading: "https://github.com/lyuwenyu/storage/releases/download/v0.1/ResNet50_vd_ssld_v2_pretrained_from_paddle.pth" to C:\Users\enriq/.cache\torch\hub\checkpoints\ResNet50_vd_ssld_v2_pretrained_from_paddle.pth


100%|██████████| 90.0M/90.0M [00:08<00:00, 11.3MB/s]


Load PResNet50 state_dict


**TRACKING Y DETECCIÓN RT-DETR**

Funciones del procesamiento de vídeo:
- **`preprocess`**: Redimensiona el fotograma a 640x640 y lo convierte en tensor, una sola vez por fotograma.
- **`make_tracker`**: Crea un tracker ByteTrack con el tiempo de memoria indicado.
- **`letterbox`**: Redimensiona el recorte manteniendo la proporción y rellena con gris, para no deformar las matrículas.
- **`run_detector`**: Ejecuta el detector y filtra por confianza y clases.
- **`detect_plates_in_crops`**: Busca la mejor matrícula dentro del recorte de cada vehículo y la devuelve en coordenadas del fotograma.

In [7]:
import supervision as sv

# el preprocesado se separa del detector para hacerlo UNA sola vez por frame
def preprocess(pil_rgb):
    im = pil_rgb.resize((640, 640), Image.BILINEAR)
    return torch.from_numpy(np.array(im)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)

def make_tracker(buffer_seconds, min_conf=0.7):
    return sv.ByteTrack(
        track_activation_threshold=min_conf,                   # score mínimo para iniciar un track
        lost_track_buffer=int(fps * buffer_seconds),      # frames que aguanta un track sin detección
        minimum_matching_threshold=0.7,
        frame_rate=int(fps),
        minimum_consecutive_frames=4,
    )

# Letterbox, mantiene la proporcion del recorte y rellena con gris
def letterbox(pil_rgb, size=640):
    w, h = pil_rgb.size
    r = size / max(w, h)                                  # escala unica: no deforma
    nw, nh = int(round(w * r)), int(round(h * r))
    canvas = Image.new("RGB", (size, size), (114, 114, 114))
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas.paste(pil_rgb.resize((nw, nh), Image.BILINEAR), (px, py))
    x = torch.from_numpy(np.array(canvas)).permute(2, 0, 1).float().div(255)[None].to(DEVICE)
    return x, r, px, py

@torch.no_grad()
def run_detector(x, mdl, post, conf, valid_classes, sizes):
    # x: tensor (B,3,640,640) ya preprocesado | sizes: tensor (B,2) con (w,h) originales
    labels, boxes, scores = post(mdl(x), sizes)
    labels, boxes, scores = labels.cpu().numpy(), boxes.cpu().numpy(), scores.cpu().numpy()
    results = []
    for l, b, s in zip(labels, boxes, scores):
        keep = (s >= conf) & np.isin(l, np.array(valid_classes))
        results.append((b[keep], s[keep], l[keep]))
    return results   # lista con una tupla (boxes, scores, labels) por cada imagen del batch

# Devuelve {tracker_id_vehiculo: (score, caja_en_frame)} con la mejor matricula de cada vehiculo
def detect_plates_in_crops(rgb, veh_xyxy, veh_tids, conf):
    H, W = rgb.shape[:2]
    if len(veh_xyxy) == 0:
        return {}

    areas = (veh_xyxy[:, 2] - veh_xyxy[:, 0]) * (veh_xyxy[:, 3] - veh_xyxy[:, 1])
    order = np.argsort(-areas)[:MAX_CROPS]          # los vehiculos mas grandes primero

    xs, meta = [], []
    for i in order:
        x1, y1, x2, y2 = veh_xyxy[i]
        bw, bh = x2 - x1, y2 - y1
        if bw < MIN_CROP_SIDE or bh < MIN_CROP_SIDE:
            continue
        mx, my = bw * CROP_MARGIN, bh * CROP_MARGIN
        cx1, cy1 = int(max(0, x1 - mx)), int(max(0, y1 - my))
        cx2, cy2 = int(min(W, x2 + mx)), int(min(H, y2 + my))
        if cx2 - cx1 < 2 or cy2 - cy1 < 2:
            continue
        x, r, px, py = letterbox(Image.fromarray(rgb[cy1:cy2, cx1:cx2]))
        xs.append(x)
        meta.append((int(veh_tids[i]), cx1, cy1, r, px, py, (x1, y1, x2, y2)))

    if not xs:
        return {}

    sizes = torch.tensor([[640, 640]] * len(xs), device=DEVICE)
    results = run_detector(torch.cat(xs), model_plate, postprocessor_plate,
                           conf, list(PLATE_CLASS), sizes)

    best = {}   # tid_vehiculo -> (score, caja)
    for (b, s, l), (tid, ox, oy, r, px, py, vb) in zip(results, meta):
        for bb, ss in zip(b, s):
            # deshacer letterbox y volver a coordenadas del frame
            box = np.array([(bb[0] - px) / r + ox, (bb[1] - py) / r + oy,
                            (bb[2] - px) / r + ox, (bb[3] - py) / r + oy])
            cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
            if not (vb[0] <= cx <= vb[2] and vb[1] <= cy <= vb[3]):
                continue   # matricula del margen que pertenece a otro coche
            if tid not in best or ss > best[tid][0]:
                best[tid] = (float(ss), box)
    return best

**PROCESAMIENTO DE VÍDEO**

Se recorre el vídeo fotograma a fotograma: se detectan personas y vehículos, se les asigna un identificador, se busca la matrícula de cada vehículo y se lee con el VLM (con control de frecuencia e intentos). Se guardan el vídeo anotado (`resultado.mp4`) y un CSV con todos los resultados (`resultado.csv`) en `RT-DETR_VLM_result`.

Antes los resultados iban a `RT-DETR_PaddleOCR_result`, la misma carpeta del cuaderno de PaddleOCR, y se sobrescribían entre sí.

La liberación de recursos está en una celda aparte, para poder ejecutarla si se interrumpe el bucle.

In [8]:
import csv
from collections import defaultdict

path_test = 'C0142.MP4'
output_dir = os.path.abspath('RT-DETR_VLM_result')
os.makedirs(output_dir, exist_ok=True)
output_path =  os.path.join(output_dir, 'resultado.mp4')
output_csv = os.path.join(output_dir, 'resultado.csv')
vid = cv2.VideoCapture(path_test)

w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = vid.get(cv2.CAP_PROP_FPS) or 30

# escritor de video a disco (sustituye a la ventana)
writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# csv con separador punto y coma
csv_file = open(output_csv, 'w', newline='', encoding='utf-8-sig')
csv_writer = csv.writer(csv_file, delimiter=';')
csv_writer.writerow(['fotograma', 'tipo_objeto', 'confianza', 'identificador_tracking',
                     'x1', 'y1', 'x2', 'y2',
                     'matricula_en_su_caso', 'confianza_matricula', 'mx1', 'my1', 'mx2', 'my2',
                     'texto_matricula'])

tracker_person = make_tracker(4)    # peatones: aguantan 4 s tapados
tracker_vehicle = make_tracker(1.5) # 3s
# tracker_plate no existe ya que la matricula hereda el id del vehiculo

size_full = torch.tensor([[w, h]], device=DEVICE)

ids_propios = {c: {} for c in CLASS_NAMES}
class_scores = defaultdict(lambda: defaultdict(float))  # key -> {clase: score acumulado}
track_class = {}                                         # key -> clase votada actual
next_num = {c: 1 for c in CLASS_NAMES}           # siguiente número visible por clase

plates_seen = set()      # vehiculos que han tenido matricula alguna vez
last_plate = {}          # tid_vehiculo -> (score, caja, frame_idx)

# Estructuras de tracking y lectura OCR
ultimo_ocr = {}                 # tid_vehiculo -> frame_idx del último intento OCR
intentos_ocr = defaultdict(int) # tid_vehiculo -> número de intentos OCR realizados
lecturas = {}                   # tid_vehiculo -> (score_total, texto_matricula)

frame_idx = 0

while vid.isOpened():
    ret, image = vid.read()

    if not ret:
      break

    # conversion y resize UNA sola vez para el detector de vehiculos
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    x_full = preprocess(Image.fromarray(rgb))

    boxes, scores, labels = run_detector(x_full, model, postprocessor, 0.5, list(BASE_CLASS), size_full)[0]    #NO TOCAR CONF(0.65)

    detections = sv.Detections(
        xyxy=boxes.astype(np.float32),
        confidence=scores.astype(np.float32),
        class_id=labels.astype(int),
    )

    det_person = detections[detections.class_id == list(BASE_CLASS)[0]]
    det_veh = detections[np.isin(detections.class_id,  list(BASE_CLASS)[1:])]
    det_veh = det_veh.with_nms(threshold=0.55, class_agnostic=False)  # duplicados de la misma clase
    det_veh = det_veh.with_nms(threshold=0.65, class_agnostic=True)    # (0.35) mismo objeto con dos clases

    det_person = tracker_person.update_with_detections(det_person)
    det_veh = tracker_vehicle.update_with_detections(det_veh)

    # matriculas: se buscan en el recorte de cada vehiculo trackeado y se asocian a su tracker_id
    plate_by_tid = detect_plates_in_crops(rgb, det_veh.xyxy, det_veh.tracker_id, PLATE_CONF)
    for tid, (sc, bx) in plate_by_tid.items():
        last_plate[tid] = (sc, bx, frame_idx)

        # Inferencia OCR sobre la placa detectada (control de frecuencia y número de intentos)
        ancho_placa = bx[2] - bx[0]
        if (ancho_placa >= MIN_ANCHO_OCR
                and intentos_ocr[tid] < OCR_MAX_INTENTOS
                and lecturas.get(tid, (0.0, ""))[0] < OCR_SCORE_OK
                and (tid not in ultimo_ocr or frame_idx - ultimo_ocr[tid] >= OCR_CADA)):
            ultimo_ocr[tid] = frame_idx
            intentos_ocr[tid] += 1
            rec_placa = recortar(image, (int(bx[0]), int(bx[1]), int(bx[2]), int(bx[3])), margen_px=4)
            texto, oconf = leer_placa(rec_placa)
            if len(texto) >= MIN_CHARS_PLACA and matricula_valida(texto):
                score_ocr = sc * oconf
                if tid not in lecturas or score_ocr > lecturas[tid][0]:
                    lecturas[tid] = (score_ocr, texto)

    detections = sv.Detections.merge([det_person, det_veh])

    frame_rows = []   # datos del frame para el csv

    for box, score, label, tid in zip(detections.xyxy, detections.confidence, detections.class_id, detections.tracker_id):
        # Los dos trackers generan IDs desde 1, así que la clave incluye el grupo
        grupo = 'person' if label == 0 else 'veh'
        key = (grupo, tid)

        # Voto: suma el score de la clase detectada en este frame
        class_scores[key][label] += score
        cls = max(class_scores[key], key=class_scores[key].get)

        # Si la clase votada cambió, mueve el track a la nueva clase (para no inflar el conteo)
        if track_class.get(key) != cls:
            old = track_class.get(key)
            if old is not None:
                ids_propios[old].pop(key, None)
            ids_propios[cls][key] = next_num[cls]
            next_num[cls] += 1
            track_class[key] = cls

        x1, y1, x2, y2 = box.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), BASE_COLORS[cls], 2)
        cv2.putText(image, f" id: {ids_propios[cls][key]} {CLASS_NAMES[cls]}: {score:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, BASE_COLORS[cls], 2)

        frame_rows.append({
            'grupo': grupo, 'cls': cls, 'score': float(score), 'tid': tid,
            'id': ids_propios[cls][key], 'box': (int(x1), int(y1), int(x2), int(y2)),
            'plate': None,
        })

    # escritura del csv (personas y vehiculos; la matricula va en la fila de su vehiculo)
    for r in frame_rows:
        if r['grupo'] == 'veh' and r['tid'] in last_plate:
            sc, bx, f = last_plate[r['tid']]
            if frame_idx - f <= HOLD_FRAMES:
                pb = tuple(int(v) for v in bx)
                txt = lecturas[r['tid']][1] if r['tid'] in lecturas else ""
                r['plate'] = {'id': r['id'], 'score': sc, 'box': pb, 'texto': txt}
                plates_seen.add(('veh', r['tid']))
                # Renderizado de matrícula idéntico a YOLO (pone '?' si no hay lectura válida)
                px1, py1, px2, py2 = pb
                cv2.rectangle(image, (px1, py1), (px2, py2), COLOR_PLACA, 2)
                txt_display = txt if txt else "?"
                fuente, esc, grosor = cv2.FONT_HERSHEY_SIMPLEX, ESCALA_TEXTO_PLACA, 1
                (tw, th), base = cv2.getTextSize(txt_display, fuente, esc, grosor)
                bx1 = max(0, px1)
                by2 = max(th + base + 6, py1 - 2)
                by1 = by2 - th - base - 6
                cv2.rectangle(image, (bx1, by1), (bx1 + tw + 8, by2), COLOR_PLACA, -1)
                cv2.putText(image, txt_display, (bx1 + 4, by2 - base - 3),
                            fuente, esc, (0, 0, 0), grosor)

        p = r['plate']
        plate_cols = ([p['id'], f"{p['score']:.4f}", *p['box'], p['texto']] if p
                      else ['', '', '', '', '', '', ''])
        csv_writer.writerow([frame_idx, CLASS_NAMES[r['cls']], f"{r['score']:.4f}", r['id'],
                             *r['box'], *plate_cols])

    # Barra inferior con los totales
    cv2.rectangle(image, (0, h - 40), (w, h), (0, 0, 0), -1)
    texto = "  |  ".join(f"{CLASS_NAMES[c]}: {len(ids_propios[c])}" for c in CLASS_NAMES)
    texto += f"  |  plate: {len(plates_seen)}"
    texto += f"  |  ocr: {len(lecturas)}"
    cv2.putText(image, texto, (10, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    writer.write(image)
    frame_idx += 1


vid.release()
writer.release()
csv_file.close()

print(f"Video guardado en {output_path} | CSV guardado en {output_csv}")
print(f"Matrículas detectadas: {len(plates_seen)} | Matrículas leídas con OCR: {len(lecturas)}")

KeyboardInterrupt: 

In [ ]:
import csv, json, time
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor


# ---------- Modelo VLM ----------
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO = torch.bfloat16 if DISPOSITIVO == "cuda" else torch.float32
ID_MODELO = "PaddlePaddle/PaddleOCR-VL"

vlm = AutoModelForCausalLM.from_pretrained(
    ID_MODELO, trust_remote_code=True, torch_dtype=TIPO).to(DISPOSITIVO).eval()
procesador = AutoProcessor.from_pretrained(ID_MODELO, trust_remote_code=True)

# ---------- Configuración de la evaluación ----------
BASE = Path("datos")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"

MAX_IMAGENES = 200
GT_KEYS = ("lpn", "text", "plate", "label", "number", "plate_number")
CONF_EVAL = 0.25
IOU_MATCH = 0.5
PAD = 0.05
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")
DIR_ERRORES = SALIDA / "errores"
DIR_ERRORES.mkdir(parents=True, exist_ok=True)

# Tensor constante para el detector (se crea una sola vez)
TAM_DET = torch.tensor([[640, 640]], device=DISPOSITIVO)

# Levenshtein: usa rapidfuzz (mucho más rápido) si está instalado
try:
    from rapidfuzz.distance import Levenshtein as _Lev
    lev = _Lev.distance
except ImportError:
    def lev(a, b):
        """Distancia de Levenshtein entre dos cadenas."""
        if a == b:
            return 0
        previa = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            actual = [i]
            for j, cb in enumerate(b, 1):
                actual.append(min(previa[j] + 1, actual[j - 1] + 1, previa[j - 1] + (ca != cb)))
            previa = actual
        return previa[-1]


# ---------- Utilidades ----------
def recortar_eval(img, caja):
    """Recorte con margen PAD."""
    h_img, w_img = img.shape[:2]
    x1, y1, x2, y2 = caja
    mx, my = PAD * (x2 - x1), PAD * (y2 - y1)
    return img[max(0, int(y1 - my)):min(h_img, int(y2 + my)),
               max(0, int(x1 - mx)):min(w_img, int(x2 + mx))]


def cargar_gt(ruta_img, w_img, h_img):
    """Carga las cajas y textos de las anotaciones JSON."""
    ruta = ANOTACIONES / f"{ruta_img.stem}.json"
    if not ruta.exists():
        return []

    gts = []
    for obj in json.loads(ruta.read_text()):
        xc, yc, bw, bh = obj["rectangle"]
        caja = ((xc - bw / 2) * w_img, (yc - bh / 2) * h_img,
                (xc + bw / 2) * w_img, (yc + bh / 2) * h_img)
        txt = next((norm(str(obj[k])) for k in GT_KEYS if obj.get(k)), None)
        gts.append((caja, txt))
    return gts


@torch.no_grad()
def detectar_rtdetr(img):
    """Detecta matrículas con RT-DETR y devuelve cajas en píxeles originales."""
    h, w = img.shape[:2]

    # Preprocesamiento igual que en el detector RT-DETR
    x, r, px, py = letterbox(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
    cajas, scores, _ = run_detector(x, model_plate, postprocessor_plate,
                                    CONF_EVAL, list(PLATE_CLASS), TAM_DET)[0]

    preds = []
    for (a, b, c, d), conf in zip(cajas, scores):
        # Deshacer letterbox y limitar a los bordes de la imagen
        x1, x2 = (a - px) / r, (c - px) / r
        y1, y2 = (b - py) / r, (d - py) / r
        preds.append(([max(0, min(w, x1)), max(0, min(h, y1)),
                       max(0, min(w, x2)), max(0, min(h, y2))], float(conf)))
    return preds


def leer_cronometrado(crop):
    """Devuelve lectura cruda, corregida y tiempo en milisegundos."""
    if crop.size == 0:
        return "", "", 0.0

    t0 = time.perf_counter()
    res = leer_placa(crop)
    cruda = res[0] if isinstance(res, (tuple, list)) and res else (res if isinstance(res, str) else "")
    ms = (time.perf_counter() - t0) * 1000

    return cruda, corregir(cruda), ms


def fila(imagen, origen, gt, cruda, corregida, ms, det_conf, iou_v):
    return dict(imagen=imagen, origen=origen, gt=gt, ocr_crudo=cruda, ocr_corregido=corregida,
                ms=round(ms, 1), det_conf=det_conf, iou=iou_v)


def resumen(titulo, rows, col):
    """Calcula acierto exacto, por carácter y tiempos."""
    if not rows:
        print(f"{titulo}: sin resultados")
        return

    exacto = np.mean([r["gt"] == r[col] for r in rows])
    por_car = np.mean([1 - min(1, lev(r["gt"], r[col]) / max(len(r["gt"]), 1)) for r in rows])
    tiempos = [r["ms"] for r in rows]

    print(f"{titulo:<40} n={len(rows):<5} exacto={exacto:.3f} | por carácter={por_car:.3f} | "
          f"{np.mean(tiempos):.0f} ms de media ({np.median(tiempos):.0f} de mediana)")


def texto_valido(t):
    return t is not None and t != "NONE"


# ---------- Evaluación ----------
leer_cronometrado(np.zeros((30, 100, 3), np.uint8))  # calentamiento del OCR

lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
if MAX_IMAGENES:
    lista = lista[:MAX_IMAGENES]

filas = []
n_gt = n_gt_texto = n_detectadas = 0

for ruta_img in lista:
    img = cv2.imread(str(ruta_img))
    if img is None:
        continue

    h_img, w_img = img.shape[:2]
    gts = cargar_gt(ruta_img, w_img, h_img)
    n_gt += len(gts)
    n_gt_texto += sum(t is not None for _, t in gts)

    preds = detectar_rtdetr(img)
    n_detectadas += len(preds)

    # 1) OCR sobre las detecciones de RT-DETR (solo si hay anotación con la que compararlas)
    for caja, conf_det in (preds if gts else ()):
        # Anotación con mayor IoU (la primera en caso de empate)
        mejor_iou, gt_txt = 0.0, None
        for gcaja, gtxt in gts:
            v = iou(caja, gcaja)
            if v > mejor_iou:
                mejor_iou, gt_txt = v, gtxt

        # Descarta antes de hacer OCR (era lo más costoso y se tiraba el resultado)
        if mejor_iou < IOU_MATCH or not texto_valido(gt_txt):
            continue

        recorte = recortar_eval(img, caja)
        cruda, corregida, ms = leer_cronometrado(recorte)
        filas.append(fila(ruta_img.name, "deteccion", gt_txt, cruda, corregida, ms,
                          round(conf_det, 3), round(mejor_iou, 3)))

        if gt_txt != cruda:
            cv2.imwrite(str(DIR_ERRORES / f"{len(filas):05d}_gt-{gt_txt}_ocr-{cruda}.jpg"), recorte)

    # 2) OCR sobre el recorte exacto de la anotación (mide el OCR sin depender del detector)
    for gcaja, gtxt in gts:
        if not texto_valido(gtxt):
            continue
        cruda, corregida, ms = leer_cronometrado(recortar_eval(img, gcaja))
        filas.append(fila(ruta_img.name, "gt_recorte", gtxt, cruda, corregida, ms, None, 1.0))


# ---------- Guardar y resumir ----------
ruta_csv = SALIDA / "rtdetr_resultados_vlm.csv"

with open(ruta_csv, "w", newline="", encoding="utf-8") as f:
    escritor_csv = csv.DictWriter(f, fieldnames=list(filas[0]) if filas else ["vacio"])
    escritor_csv.writeheader()
    escritor_csv.writerows(filas)

print(f"\nImágenes: {len(lista)} | placas anotadas: {n_gt} | detecciones RT-DETR: {n_detectadas}")

if n_gt_texto == 0:
    print(f"Las anotaciones no traen texto (claves probadas: {GT_KEYS}).")
else:
    det = [r for r in filas if r["origen"] == "deteccion"]
    techo = [r for r in filas if r["origen"] == "gt_recorte"]

    resumen("VLM sobre detecciones RT-DETR", det, "ocr_crudo")
    resumen("VLM sobre recorte anotado (techo)", techo, "ocr_crudo")
    resumen("VLM + corrección, detecciones", det, "ocr_corregido")
    resumen("VLM + corrección, recorte anotado", techo, "ocr_corregido")

    print(f"Placas anotadas encontradas por RT-DETR: {len(det)}/{n_gt_texto}")

print(f"\nCSV: {ruta_csv}")
print(f"Recortes con error: {DIR_ERRORES}")


Imágenes: 200 | placas anotadas: 411 | detecciones RT-DETR: 516
VLM sobre detecciones RT-DETR            n=198   exacto=0.803 | por carácter=0.931 | 1045 ms de media (568 de mediana)
VLM sobre recorte anotado (techo)        n=208   exacto=0.774 | por carácter=0.917 | 1032 ms de media (566 de mediana)
VLM + corrección, detecciones            n=198   exacto=0.838 | por carácter=0.934 | 1045 ms de media (568 de mediana)
VLM + corrección, recorte anotado        n=208   exacto=0.808 | por carácter=0.922 | 1032 ms de media (566 de mediana)
Placas anotadas encontradas por RT-DETR: 198/411

CSV: ocr_eval_rtdetr\ocr_resultados_rtdetr.csv
Recortes con error: ocr_eval_rtdetr\errores
